In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Simulare exactă și zgomotoasă cu primitivele Qiskit Aer

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
[Simularea exactă cu primitivele Qiskit](/guides/simulate-with-qiskit-sdk-primitives) demonstrează cum să folosești primitivele de referință incluse în Qiskit pentru a efectua simularea exactă a circuitelor cuantice. Procesoarele cuantice existente în prezent sunt afectate de erori, sau zgomot, astfel că rezultatele unei simulări exacte nu reflectă neapărat rezultatele pe care le-ai obține când rulezi circuite pe hardware real. Deși primitivele de referință din Qiskit nu suportă modelarea zgomotului, [Qiskit Aer](https://qiskit.org/ecosystem/aer/) include implementări ale primitivelor care suportă modelarea zgomotului. Qiskit Aer este un simulator de circuite cuantice de înaltă performanță pe care îl poți folosi în locul primitivelor de referință pentru performanță mai bună și mai multe funcționalități. Face parte din [Ecosistemul Qiskit](https://qiskit.github.io/ecosystem/). În acest articol, demonstrăm utilizarea primitivelor Qiskit Aer pentru simulare exactă și zgomotoasă.

> **Note:** - Este necesară versiunea `qiskit-aer` v0.14 sau mai recentă.
> - Deși primitivele Qiskit Aer implementează interfețele primitive, ele nu oferă aceleași opțiuni ca primitivele Qiskit Runtime. Nivelul de reziliență, de exemplu, nu este disponibil cu primitivele Qiskit Aer.
> - Consultă [documentația AerSimulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) pentru detalii despre opțiunile metodei de simulare pe care Aer le suportă.

Pentru a explora simularea exactă și zgomotoasă, creează un circuit exemplu pe opt qubiți:

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg)

Acest Circuit conține parametri pentru a reprezenta unghiurile de rotație pentru Gate-urile $R_y$ și $R_z$. La simularea acestui Circuit, trebuie să specificăm valori explicite pentru acești parametri. În celula următoare, specificăm câteva valori pentru acești parametri și folosim primitiva Estimator din Qiskit Aer pentru a calcula valoarea exactă a valorii așteptate a observabilului $ZZ \cdots Z$.

In [2]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = Estimator()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.8870140234256602

Now, let's initialize a noise model that includes depolarizing error of 2% on every CX gate. In practice, the error arising from the two-qubit gates, which are CX gates here, are the dominant source of error when running a circuit. See [Build noise models](./build-noise-models) for an overview of constructing noise models in Qiskit Aer.

In the next cell, we construct an Estimator that incorporates this noise model and use it to compute the expectation value of the observable.

In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)

noisy_estimator = Estimator(
    options=dict(backend_options=dict(noise_model=noise_model))
)
job = noisy_estimator.run([pub])
result = job.result()
pub_result = result[0]
noisy_value = float(pub_result.data.evs)
noisy_value

0.7247404214143528

Acum, să inițializăm un model de zgomot care include eroare de depolarizare de 2% pe fiecare Gate CX. În practică, eroarea care provine din Gate-urile cu doi qubiți, care sunt Gate-urile CX în acest caz, reprezintă sursa dominantă de eroare la rularea unui Circuit. Consultă [Construiește modele de zgomot](./build-noise-models) pentru o prezentare generală a construirii modelelor de zgomot în Qiskit Aer.

În celula următoare, construim un Estimator care încorporează acest model de zgomot și îl folosim pentru a calcula valoarea așteptată a observabilului.

In [4]:
cx_count = circuit.count_ops()["cx"]
(1 - cx_depolarizing_prob) ** cx_count

0.6542558123199923

This value, 65%, gives a rough estimate of the probability that our final state is correct. It is a conservative estimate because it does not take into account the initial state of the simulation.

The following code cell shows how to use the Sampler primitive from Qiskit Aer to sample from the noisy circuit. We need to add measurements to the circuit before running it with the Sampler primitive.

In [5]:
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params, 100)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'00100000': 1,
 '00000000': 65,
 '10101000': 1,
 '10000000': 5,
 '00001000': 1,
 '00000110': 2,
 '11110010': 1,
 '00000011': 3,
 '01010000': 3,
 '11000000': 3,
 '01111000': 1,
 '01000000': 2,
 '00000010': 1,
 '01100000': 1,
 '00011000': 1,
 '00111100': 1,
 '00010100': 1,
 '00001111': 1,
 '00110000': 1,
 '01100101': 1,
 '00000100': 1,
 '10100000': 1,
 '00000001': 1,
 '11010000': 1}

După cum poți vedea, valoarea așteptată în prezența zgomotului este destul de departe de valoarea corectă. În practică, poți folosi o varietate de tehnici de atenuare a erorilor pentru a contracara efectele zgomotului, dar o discuție despre aceste tehnici depășește scopul acestui articol.

Pentru a obține o idee aproximativă despre cum afectează zgomotul rezultatul final, ia în considerare modelul nostru de zgomot, care adaugă o eroare de depolarizare de 2% la fiecare Gate CX. Eroarea de depolarizare cu probabilitatea $p$ este definită ca un canal cuantic $E$ care are următoarea acțiune asupra unei matrice densitate $\rho$:

$$
E(\rho) = (1 - p) \rho + p\frac{I}{2^n}
$$

unde $n$ este numărul de qubiți, în acest caz, 2. Adică, cu probabilitatea $p$, starea este înlocuită cu starea complet mixtă, iar starea este păstrată cu probabilitatea $1 - p$. După $m$ aplicări ale canalului de depolarizare, probabilitatea ca starea să fie păstrată ar fi $(1 - p)^m$. Prin urmare, ne așteptăm ca probabilitatea de a păstra starea corectă la sfârșitul simulării să scadă exponențial cu numărul de Gate-uri CX din Circuit.

Să numărăm Gate-urile CX din Circuit și să calculăm $(1 - p)^m$. Apelăm `count_ops` pentru a obține un dicționar care mapează numele Gate-urilor la numărători și recuperăm intrarea pentru Gate-ul CX.